# 04 · Validate — BindCraft vs RFdiffusion + epitope competition

**Standard slot:** *validate (in silico).* **For Project 06 this is the core comparison:** the
head-to-head between the two paradigms (hit rate, interface energy, novelty) plus **epitope-competition
reasoning vs PD-1**, with publication-style figures (D3 part 2).

Needs `results/bindcraft_designs.csv` + `results/rfdiffusion_designs.csv` + `results/all_ranked.csv`
(from notebooks 02–03).

## Setup paths

In [ ]:
import sys, os
# Make the project's scripts/ and the cohort's shared/ importable.
# Adjust these if your Colab working directory differs (see 00_setup §5 for Drive mounting).
sys.path.insert(0, os.path.abspath("../scripts"))
sys.path.insert(0, os.path.abspath("../../../shared"))
os.makedirs("results", exist_ok=True)
print("paths ready; cwd =", os.getcwd())

## 1 · Head-to-head hit rate + interface energy

Compare the two paradigms on (a) all-layers **hit rate** and (b) the **interface-energy** (`rosetta_dG`)
distribution of survivors. A fair comparison filters both identically (notebook 03) and reports the
*distribution*, not the single best. Mock numbers are SYNTHETIC.

In [ ]:
import pandas as pd, numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

ranked = pd.read_csv("results/all_ranked.csv")
print("paradigms:", ranked["paradigm"].value_counts().to_dict())

summary = []
for p, g in ranked.groupby("paradigm"):
    n = len(g); passed = int((g["layers_passed"] >= 3).sum())
    summary.append(dict(paradigm=p, n=n, all_layers_survivors=passed,
                        hit_rate_pct=round(100*passed/max(n,1), 1),
                        median_pae=round(float(g["pae_interaction"].median()), 2),
                        median_dG=round(float(g["rosetta_dG"].median()), 2)))
summary = pd.DataFrame(summary)
print("\nhead-to-head summary (SYNTHETIC if mock):")
print(summary.to_string(index=False))

In [ ]:
# Interface-energy distribution per paradigm (survivors).
fig, ax = plt.subplots(1, 2, figsize=(10, 3.6))
for p, g in ranked.groupby("paradigm"):
    surv = g[g["layers_passed"] >= 3]
    ax[0].hist(g["pae_interaction"].dropna(), bins=15, alpha=0.5, label=p)
    ax[1].hist(surv["rosetta_dG"].dropna(), bins=15, alpha=0.5, label=p)
ax[0].set_xlabel("pae_interaction (Å, lower better)"); ax[0].set_ylabel("designs"); ax[0].set_title("AF2-Multimer pae_interaction"); ax[0].legend()
ax[1].set_xlabel("rosetta_dG (REU, more negative better)"); ax[1].set_title("Interface energy (survivors)"); ax[1].legend()
fig.suptitle("BindCraft vs RFdiffusion (EXAMPLE_DATA if mock)")
plt.tight_layout(); plt.savefig("results/p06_headtohead.png", dpi=150); plt.show()
print("saved results/p06_headtohead.png")

## 2 · Novelty `[extension]`

Novelty = TM-score of each binder to its nearest natural fold (Foldseek/TM-align; `< 0.5` ≈ novel).
On Colab, compute it per design and compare the two paradigms' novelty distributions. Here we scaffold
the analysis (mock has no real structures), so we just show where it plugs in.

In [ ]:
# Scaffold: on Colab, run Foldseek/TM-align on each predicted binder backbone -> tm_to_pdb,
# then compare distributions across paradigms (novel == tm_to_pdb < 0.5).
# Example shape of the analysis once tm_to_pdb is populated:
if "tm_to_pdb" in ranked.columns and ranked["tm_to_pdb"].notna().any():
    for p, g in ranked.groupby("paradigm"):
        novel = (g["tm_to_pdb"] < 0.5).mean()
        print(f"{p:12s}: novel fraction (TM<0.5) = {novel:.2f}")
else:
    print("Novelty scaffold — populate tm_to_pdb with Foldseek/TM-align on Colab, then compare paradigms.")

## 3 · Epitope competition vs PD-1 `[extension]`

A binder only **blocks** PD-1 if it covers the PD-1 footprint. `hotspot_overlap` (notebook 02) is our
geometry proxy: the fraction of PD-1-face hotspots the binder contacts. Higher ⇒ more likely a
competitive blocker. Compare the survivors' overlap across paradigms — a strong interface that *misses*
the PD-1 face is not a checkpoint blocker.

In [ ]:
bc = pd.read_csv("results/bindcraft_designs.csv")
rf = pd.read_csv("results/rfdiffusion_designs.csv")
pools = pd.concat([bc, rf], ignore_index=True)

# Join overlap onto the ranked survivors.
ov = pools.set_index("design_id")["hotspot_overlap"]
ranked["hotspot_overlap"] = ranked["design_id"].map(ov)
surv = ranked[ranked["layers_passed"] >= 3]

print("epitope-competition overlap of all-layers survivors (SYNTHETIC if mock):")
for p, g in surv.groupby("paradigm"):
    print(f"  {p:12s}: median PD-1-footprint overlap = {g['hotspot_overlap'].median():.2f}  (n={len(g)})")

# "Competitive blockers" = survivors that also cover enough of the PD-1 footprint.
BLOCK_OVERLAP = 0.5
blockers = surv[surv["hotspot_overlap"] >= BLOCK_OVERLAP]
print(f"\nlikely competitive blockers (survivor AND overlap>={BLOCK_OVERLAP}): {len(blockers)}")
print(blockers.groupby("paradigm").size().to_dict())

## 4 · Select the top 10–20 per paradigm

The D★ deliverable wants the **top 10–20 each**. Rank survivors by the composite score and, as a
tie-breaker for *blockers*, prefer higher PD-1-footprint overlap. Save the shortlist for the validation
plan (notebook 05).

In [ ]:
top_per = []
for p, g in ranked.groupby("paradigm"):
    g2 = g[g["layers_passed"] >= 3].sort_values(
        ["score", "hotspot_overlap"], ascending=False).head(20)
    top_per.append(g2)
top = pd.concat(top_per, ignore_index=True)
top.to_csv("results/top_candidates.csv", index=False)
print("wrote results/top_candidates.csv:", top.shape, "(top<=20 per paradigm)")
print(top.groupby("paradigm").size().to_dict())
top.head(8)[["design_id", "paradigm", "score", "pae_interaction", "rosetta_dG", "hotspot_overlap"]]

## D3 (part 2) checklist
- [ ] Head-to-head: hit rate + interface-energy distribution per paradigm (figure `results/p06_headtohead.png`).
- [ ] Novelty compared across paradigms (TM-score to PDB) — or the scaffold wired up on Colab.
- [ ] Epitope-competition vs PD-1: footprint overlap of survivors; "competitive blocker" count.
- [ ] `results/top_candidates.csv`: top 10–20 each, ready for the validation plan.
- [ ] Honest discussion of the two paradigms' different failure modes (not just a winner).

**Next:** `05_validation_plan.ipynb` — the SPR/BLI + PD-1-competition plan.